# Predict ALL labels from autoencoder embeddings

Same idea as `predict_from_embeddings.ipynb` — frozen encoder from the
autoencoder checkpoint, small MLP head per label — but loops over **every**
label found in the label JSONs while sharing all other settings:

- embeddings are computed **once** and reused for all labels
- the train/val split is drawn once from `SEED`, so every label is scored on
  the same validation plots (comparable metrics)
- each label gets its own head, trained with the same
  `BATCH_SIZE`/`EPOCHS`/`LR`/`PATIENCE`/`MIN_DELTA`, and its weights (the
  best-validation one when a val split exists) plus the label
  standardization stats are saved to `HEADS_DIR/<LABEL_KEY>.pt`
- a `metrics.json` summary (RMSE/MAE/R² per label on the val split) is
  written to `HEADS_DIR`

The last cell makes the same graphics as the single-label notebook for
comparison: a grid of true-vs-predicted scatters (one per label), a bar chart
of per-label RMSE (normalized by the label's train std, since raw units
differ) and R², and the embedding-norm histogram (shared by all labels).

Requires: `torch`, `numpy` (and `matplotlib` only for the optional plots).

In [ ]:
import glob
import json
import os

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset

# ---- device: CPU / Apple integrated GPU (MPS) / CUDA cores ---------------
DEVICE_CHOICE = "auto"  # "auto" | "cpu" | "mps" | "cuda"


def pick_device(choice: str) -> torch.device:
    if choice in ("cuda", "auto") and torch.cuda.is_available():
        return torch.device("cuda")
    if choice in ("mps", "auto") and torch.backends.mps.is_available():
        return torch.device("mps")  # Apple integrated GPU
    if choice != "auto":
        print(f"warning: {choice!r} not available, falling back to CPU")
    return torch.device("cpu")


device = pick_device(DEVICE_CHOICE)
print("device:", device)

In [ ]:
# ---- user config -----------------------------------------------------------
FEATURES_DIR = "../dl_dataset/features"  # prepare.py -o <out>/features
LABELS_DIR = "../dl_dataset/labels"      # prepare.py -o <out>/labels
EXTRA_DIMS = ["semantic_code"]           # MUST match the autoencoder's EXTRA_DIMS
CKPT_PATH = "../checkpoints/autoencoder.pt"  # checkpoint written by autoencoder.ipynb
HEADS_DIR = "../checkpoints/heads"       # one <LABEL_KEY>.pt per label + metrics.json
N_POINTS = 1024     # point count per cloud (encoder is N-agnostic; match training)
N_SAMPLES = "all"   # how many plots to load: an int, or "all"
EMBED_BATCH = 32    # plots per encoder forward when embedding (GPU memory guard:
                    # the point-MLP intermediate is (B, 1024, N_POINTS))
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-3
DROPOUT = 0.3    # dropout probability in the heads (0 = off)
VAL_FRAC = 0.2
SEED = 42
STANDARDIZE_LABELS = True  # z-score each label using train-set stats
PATIENCE = None     # early stopping per label on the val loss; None = off
MIN_DELTA = 0.0     # minimum val-loss improvement that resets the patience counter

_first_feat = sorted(glob.glob(os.path.join(FEATURES_DIR, "*.npz")))
_first_lab = sorted(glob.glob(os.path.join(LABELS_DIR, "*.json")))
if not _first_feat or not _first_lab:
    raise FileNotFoundError("run prepare.py first - no feature/label files found")
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"no checkpoint at {CKPT_PATH!r} - run autoencoder.ipynb "
                            f"first (its CKPT_PATH)")
with np.load(_first_feat[0]) as z:
    print("available feature arrays:", z.files)
with open(_first_lab[0]) as fh:
    print("available labels:", sorted(json.load(fh)["labels"].keys()))
os.makedirs(HEADS_DIR, exist_ok=True)

In [ ]:
class PlotDataset(Dataset):
    """One plot per sample: a (C, N_POINTS) feature matrix from the .npz."""

    def __init__(self, features_dir, extra_dims, n_points, n_samples=None):
        self.n_points = n_points
        self.extra_dims = list(extra_dims)
        feats = sorted(glob.glob(os.path.join(features_dir, "*.npz")))
        if n_samples is not None:
            feats = feats[:n_samples]
        if not feats:
            raise FileNotFoundError(f"no feature files in {features_dir!r}")
        self.feats = feats
        self.names = [os.path.splitext(os.path.basename(f))[0] for f in feats]
        with np.load(feats[0]) as z:
            available = set(z.files)
        missing = [k for k in self.extra_dims if k not in available]
        if missing:
            raise KeyError(f"feature dims {missing} not in the .npz; "
                           f"available: {sorted(available)}")

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, i):
        with np.load(self.feats[i]) as z:
            cols = [z["points"].astype(np.float32)]          # (N, 3)
            for k in self.extra_dims:
                cols.append(z[k].astype(np.float32)[:, None])  # (N, 1)
        mat = np.concatenate(cols, axis=1)                   # (N, C)
        n = len(mat)
        if n >= self.n_points:
            mat = mat[: self.n_points]
        else:
            reps = int(np.ceil(self.n_points / n))
            mat = np.tile(mat, (reps, 1))[: self.n_points]
        return torch.from_numpy(mat.T.copy())                 # (C, N_POINTS)


dataset = PlotDataset(FEATURES_DIR, EXTRA_DIMS, N_POINTS,
                      n_samples=None if N_SAMPLES == "all" else N_SAMPLES)
print(f"loaded {len(dataset)} sample(s): {dataset.names}")

# label table: plot name -> {label_key: value}; keep numeric labels only
label_table = {}
label_keys = None
for name in dataset.names:
    with open(os.path.join(LABELS_DIR, name + ".json")) as fh:
        labs = json.load(fh)["labels"]
    label_table[name] = labs
    if label_keys is None:
        label_keys = set(labs)
    else:
        label_keys &= set(labs)
LABEL_KEYS = []
for k in sorted(label_keys):
    try:
        vals = [float(label_table[n][k]) for n in dataset.names]
    except (TypeError, ValueError):
        print(f"skipping non-numeric label {k!r}")
        continue
    LABEL_KEYS.append(k)
print(f"training one head per label ({len(LABEL_KEYS)}): {LABEL_KEYS}")

In [ ]:
def chamfer_loss(a, b):
    """Symmetric Chamfer distance on the xyz channels. a, b: (B, C, N).
    O(N^2) memory per sample - keep N_POINTS modest."""
    pa = a[:, :3].transpose(1, 2)  # (B, N, 3)
    pb = b[:, :3].transpose(1, 2)
    d = torch.cdist(pa, pb)        # (B, N, N)
    return (d.min(dim=2).values.mean(dim=1) + d.min(dim=1).values.mean(dim=1)).mean()


def bound_xyz(out, output_range):
    """Bound the decoder's xyz channels (first 3) to output_range; extra dims
    pass through linearly."""
    if output_range == "neg1_1":
        return torch.cat([torch.tanh(out[:, :3]), out[:, 3:]], dim=1)
    if output_range == "zero_1":
        return torch.cat([torch.sigmoid(out[:, :3]), out[:, 3:]], dim=1)
    return out


class PointNetAE(nn.Module):
    """Bare-bones PointNet autoencoder: shared point MLP -> global max pool ->
    FC encoder to a latent vector -> FC decoder back to (C, N)."""

    def __init__(self, in_channels, n_points, latent_dim, output_range="none"):
        super().__init__()
        self.in_channels = in_channels
        self.n_points = n_points
        self.output_range = output_range
        self.enc = nn.Sequential(
            nn.Conv1d(in_channels, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(DROPOUT),
        )
        self.enc_fc = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, n_points * in_channels),
        )

    def encode(self, x):  # (B, C, N) -> latent (B, L)
        return self.enc_fc(torch.max(self.enc(x), dim=2).values)

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        z = self.encode(x)
        out = self.dec_fc(z).view(-1, self.in_channels, self.n_points)
        return bound_xyz(out, self.output_range), z


# ---- PointNet++ building blocks ---------------------------------------------
def index_points(points, idx):
    """points: (B, N, C); idx: (B, S) or (B, S, G) -> (B, S, C) or (B, S, G, C)."""
    if idx.dim() == 2:
        idx = idx.unsqueeze(-1).expand(*idx.shape, points.shape[-1])
        return torch.gather(points, 1, idx)
    points = points.unsqueeze(1).expand(-1, idx.shape[1], -1, -1)
    idx = idx.unsqueeze(-1).expand(*idx.shape, points.shape[-1])
    return torch.gather(points, 2, idx)


def farthest_point_sample(xyz, npoint):
    """Iterative FPS. xyz: (B, N, 3) -> LongTensor (B, npoint).
    Python loop over npoint - fine for a few hundred samples."""
    B, N, _ = xyz.shape
    device = xyz.device
    idx = torch.zeros(B, npoint, dtype=torch.long, device=device)
    dist = torch.full((B, N), 1e10, device=device)
    farthest = torch.randint(0, N, (B,), device=device)
    batch = torch.arange(B, device=device)
    for i in range(npoint):
        idx[:, i] = farthest
        centroid = xyz[batch, farthest, :].unsqueeze(1)  # (B, 1, 3)
        d = ((xyz - centroid) ** 2).sum(-1)
        dist = torch.min(dist, d)
        farthest = dist.argmax(dim=1)  # (B,) farthest point per batch row
    return idx


def ball_query(radius, nsample, xyz, new_xyz):
    """Ball query: up to nsample points within radius of each centroid;
    centroids with fewer neighbors repeat their nearest point.
    xyz: (B, N, 3), new_xyz: (B, S, 3) -> idx (B, S, nsample)."""
    d = torch.cdist(new_xyz, xyz)                       # (B, S, N)
    order = d.argsort(dim=-1)                           # nearest first
    idx = order[:, :, :nsample]                         # (B, S, nsample)
    in_ball = (torch.gather(d, 2, idx) <= radius)       # (B, S, nsample)
    # replace out-of-ball picks with the nearest point (order[:, :, 0])
    first = order[:, :, :1].expand(-1, -1, nsample)
    return torch.where(in_ball, idx, first)


class SetAbstraction(nn.Module):
    """One PointNet++ set-abstraction level: FPS sample -> ball-query group ->
    shared point MLP on (relative xyz + features) -> max pool per group."""

    def __init__(self, npoint, radius, nsample, in_ch, mlp):
        super().__init__()
        self.npoint, self.radius, self.nsample = npoint, radius, nsample
        layers = []
        ch = in_ch + 3  # relative xyz is concatenated to the point features
        for w in mlp:
            layers += [nn.Conv2d(ch, w, 1), nn.BatchNorm2d(w), nn.ReLU(), nn.Dropout(DROPOUT)]
            ch = w
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, points):
        # xyz: (B, N, 3); points: (B, C, N)
        new_xyz = index_points(xyz, farthest_point_sample(xyz, self.npoint))  # (B, S, 3)
        idx = ball_query(self.radius, self.nsample, xyz, new_xyz)            # (B, S, G)
        grouped_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)           # (B, S, G, 3)
        grouped_feat = index_points(points.transpose(1, 2), idx)              # (B, S, G, C)
        new_points = torch.cat([grouped_xyz, grouped_feat], dim=-1)
        new_points = self.mlp(new_points.permute(0, 3, 2, 1))                 # (B, W, G, S)
        new_points = torch.max(new_points, dim=2).values                      # (B, W, S)
        return new_xyz, new_points


class FeaturePropagation(nn.Module):
    """One PointNet++ feature-propagation level: k-NN inverse-distance
    interpolation of the coarse features -> concat skip connection -> MLP."""

    def __init__(self, in_ch, mlp):
        super().__init__()
        layers = []
        ch = in_ch
        for w in mlp:
            layers += [nn.Conv1d(ch, w, 1), nn.BatchNorm1d(w), nn.ReLU(), nn.Dropout(DROPOUT)]
            ch = w
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, new_xyz, points, new_points):
        # xyz: (B, N, 3) target level; new_xyz: (B, M, 3) source level
        # points: (B, C1, N) skip features; new_points: (B, C2, M) coarse features
        M = new_xyz.shape[1]
        if M == 1:
            interp = new_points.expand(-1, -1, xyz.shape[1])
        else:
            k = min(3, M)                                    # 3-NN (fewer if M < 3)
            d = torch.cdist(new_xyz, xyz)                    # (B, M, N)
            dist, idx = d.topk(k, dim=1, largest=False)      # k-NN of each target
            weight = 1.0 / (dist + 1e-8)
            weight = weight / weight.sum(dim=1, keepdim=True)
            interp = (index_points(new_points.transpose(1, 2), idx)
                      * weight.unsqueeze(-1)).sum(dim=1)        # (B, N, C2)
            interp = interp.transpose(1, 2)                  # (B, C2, N)
        new_points = torch.cat([interp, points], dim=1) if points is not None else interp
        return self.mlp(new_points)


class PointNetPP_AE(nn.Module):
    """PointNet++ autoencoder: two set-abstraction levels -> global max pool ->
    FC latent -> FC back to the coarsest level -> feature-propagation decoder
    with skip connections -> per-point head. Decoder xyz output is bounded to
    `output_range`."""

    def __init__(self, in_channels, n_points, latent_dim, output_range="none",
                 sa_points=(512, 128), sa_radii=(0.2, 0.4), sa_nsample=32):
        super().__init__()
        if len(sa_points) != len(sa_radii):
            raise ValueError("sa_points and sa_radii must have the same length")
        if max(sa_points) > n_points:
            raise ValueError(f"max(SA_POINTS)={max(sa_points)} exceeds "
                             f"N_POINTS={n_points}")
        if output_range not in ("none", "neg1_1", "zero_1"):
            raise ValueError(f"output_range must be none|neg1_1|zero_1, got {output_range!r}")
        self.in_channels = in_channels
        self.n_points = n_points
        self.output_range = output_range
        c0, c1 = 128, 256
        self.sa1 = SetAbstraction(sa_points[0], sa_radii[0], sa_nsample,
                                  in_channels, [64, 64, c0])
        self.sa2 = SetAbstraction(sa_points[1], sa_radii[1], sa_nsample,
                                  c0, [128, 128, c1])
        self.enc_fc = nn.Sequential(
            nn.Linear(c1, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(512, c1 * sa_points[1]),
        )
        self.fp1 = FeaturePropagation(c1 + c0, [256, 256])
        self.fp2 = FeaturePropagation(256 + in_channels, [256, 128])
        self.head = nn.Sequential(
            nn.Conv1d(128, 128, 1), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Conv1d(128, in_channels, 1),
        )

    def encode(self, x):  # (B, C, N) -> latent (B, L)
        xyz = x[:, :3].transpose(1, 2)                    # (B, N, 3)
        l1_xyz, l1_points = self.sa1(xyz, x)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        return self.enc_fc(torch.max(l2_points, dim=2).values)

    def forward(self, x):  # x: (B, C, N) -> recon (B, C, N), latent (B, L)
        xyz = x[:, :3].transpose(1, 2)                    # (B, N, 3)
        l1_xyz, l1_points = self.sa1(xyz, x)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        z = self.enc_fc(torch.max(l2_points, dim=2).values)          # (B, L)
        coarse = self.dec_fc(z).view(-1, l2_points.shape[1], l2_xyz.shape[1])
        p1 = self.fp1(l1_xyz, l2_xyz, l1_points, coarse)             # (B, 256, S1)
        p0 = self.fp2(xyz, l1_xyz, x, p1)                            # (B, 128, N)
        return bound_xyz(self.head(p0), self.output_range), z


def build_model(arch, in_channels, n_points, latent_dim, output_range,
                sa_points, sa_radii, sa_nsample):
    if arch == "pointnet":
        return PointNetAE(in_channels, n_points, latent_dim, output_range)
    if arch in ("pointnet++", "pointnetpp"):
        return PointNetPP_AE(in_channels, n_points, latent_dim, output_range,
                             tuple(sa_points), tuple(sa_radii), sa_nsample)
    raise ValueError(f"unknown ARCH {arch!r} (pointnet | pointnet++)")


def make_criterion(name):
    if name == "mse":
        return nn.MSELoss()
    if name == "l1":
        return nn.L1Loss()
    if name == "huber":
        return nn.HuberLoss()
    if name == "chamfer":
        return chamfer_loss
    raise ValueError(f"unknown LOSS {name!r} (mse | l1 | huber | chamfer)")



# ---- load the frozen encoder from the autoencoder checkpoint ---------------
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
model = build_model(ckpt["arch"], in_channels=ckpt["in_channels"],
                    n_points=ckpt["n_points"], latent_dim=ckpt["latent_dim"],
                    output_range=ckpt["output_range"], sa_points=ckpt["sa_points"],
                    sa_radii=ckpt["sa_radii"], sa_nsample=ckpt["sa_nsample"]).to(device)
model.load_state_dict(ckpt["state_dict"])
for p in model.parameters():
    p.requires_grad_(False)  # frozen encoder - only the heads train
model.eval()
LATENT_DIM = ckpt["latent_dim"]
print(f"loaded frozen encoder from {CKPT_PATH} "
      f"(arch={ckpt['arch']}, latent_dim={LATENT_DIM})")


def build_head():
    """Fresh MLP head on the latent vector, one per label."""
    return nn.Sequential(
        nn.Linear(LATENT_DIM, 256), nn.ReLU(), nn.Dropout(DROPOUT),
        nn.Linear(256, 1),
    ).to(device)

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

# Embed every plot ONCE with the frozen encoder, in chunks of EMBED_BATCH:
# (n, LATENT_DIM). One forward over ALL plots at once would build a
# (n, 1024, N_POINTS) intermediate - tens of GB on a GPU (CUDA OOM).
Zs = []
with torch.no_grad():
    for off in range(0, len(dataset), EMBED_BATCH):
        chunk = torch.stack([dataset[i] for i in
                             range(off, min(off + EMBED_BATCH, len(dataset)))])
        Zs.append(model.encode(chunk.to(device)).cpu())
Z = torch.cat(Zs)                                   # (n, LATENT_DIM)
print(f"embeddings: {tuple(Z.shape)}, finite={bool(torch.isfinite(Z).all())}")

# ONE split for all labels, so metrics are comparable across labels
n_val = max(1, int(round(len(dataset) * VAL_FRAC))) if len(dataset) > 1 else 0
perm = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(SEED))
val_idx, train_idx = perm[:n_val], perm[n_val:]

criterion = nn.MSELoss()
metrics = {}

for label_key in LABEL_KEYS:
    ys_t = torch.tensor([float(label_table[n][label_key]) for n in dataset.names],
                        dtype=torch.float32)
    # z-score this label using train-set stats only
    y_mean = ys_t[train_idx].mean()
    y_std = ys_t[train_idx].std()
    if STANDARDIZE_LABELS and float(y_std) > 0:
        y_fit = (ys_t - y_mean) / y_std
    else:
        y_mean, y_std = torch.tensor(0.0), torch.tensor(1.0)
        y_fit = ys_t.clone()

    train_loader = DataLoader(TensorDataset(Z[train_idx], y_fit[train_idx]),
                              batch_size=BATCH_SIZE, shuffle=True,
                              drop_last=len(train_idx) > BATCH_SIZE)
    val_loader = DataLoader(TensorDataset(Z[val_idx], y_fit[val_idx]),
                            batch_size=BATCH_SIZE) if n_val else None

    head = build_head()
    opt = torch.optim.Adam(head.parameters(), lr=LR)
    best_val = float("inf")
    bad = 0
    best_state = None
    stopped = None
    for epoch in range(1, EPOCHS + 1):
        head.train()
        for zb, yb in train_loader:
            zb, yb = zb.to(device), yb.to(device)[:, None]
            if zb.shape[0] == 1:
                zb = zb.repeat(BATCH_SIZE, 1)
                yb = yb.repeat(BATCH_SIZE, 1)
            opt.zero_grad()
            loss = criterion(head(zb), yb)
            loss.backward()
            opt.step()
        if val_loader is not None:
            head.eval()
            vt = 0.0
            vn = 0
            with torch.no_grad():
                for zb, yb in val_loader:
                    zb, yb = zb.to(device), yb.to(device)[:, None]
                    vt += criterion(head(zb), yb).item() * zb.shape[0]
                    vn += zb.shape[0]
            val_loss = vt / max(vn, 1)
            # always keep the best-val head (it is what gets saved);
            # PATIENCE only controls the early stop
            if val_loss < best_val - MIN_DELTA:
                best_val = val_loss
                bad = 0
                best_state = {k: v.detach().clone() for k, v in head.state_dict().items()}
            else:
                bad += 1
                if PATIENCE is not None and bad > PATIENCE:
                    stopped = epoch
                    break
    if best_state is not None:
        head.load_state_dict(best_state)

    # eval on the shared val split, in original units
    head.eval()
    eval_idx = val_idx if n_val else torch.arange(len(dataset))
    with torch.no_grad():
        pred = head(Z[eval_idx].to(device)).cpu().squeeze(-1) * y_std + y_mean
    true = ys_t[eval_idx]
    rmse = torch.sqrt(torch.mean((pred - true) ** 2)).item()
    mae = torch.mean(torch.abs(pred - true)).item()
    ss_tot = torch.sum((true - true.mean()) ** 2).item()
    # None (JSON null) when the val split has <2 points (ss_tot == 0)
    r2 = (1.0 - torch.sum((pred - true) ** 2).item() / ss_tot) if ss_tot > 0 else None
    metrics[label_key] = {"rmse": rmse, "mae": mae, "r2": r2,
                          "val_mse_standardized": best_val if n_val else None,
                          "n_val": int(len(eval_idx)), "epochs_stopped": stopped,
                          "y_mean": float(y_mean), "y_std": float(y_std)}

    # save this label's head + standardization stats
    torch.save({"label_key": label_key, "latent_dim": LATENT_DIM,
                "y_mean": float(y_mean), "y_std": float(y_std),
                "state_dict": head.state_dict()},
               os.path.join(HEADS_DIR, f"{label_key}.pt"))
    note = f" (early stop @ {stopped})" if stopped else ""
    r2_txt = f"{r2:.4f}" if r2 is not None else "n/a"
    print(f"{label_key}: RMSE {rmse:.6g}  MAE {mae:.6g}  R2 {r2_txt}  "
          f"n_val={len(eval_idx)}{note}  -> {HEADS_DIR}/{label_key}.pt")

with open(os.path.join(HEADS_DIR, "metrics.json"), "w") as fh:
    json.dump(metrics, fh, indent=2)
print(f"wrote {os.path.join(HEADS_DIR, 'metrics.json')}")

In [ ]:
ys_t_all = {k: torch.tensor([float(label_table[n][k]) for n in dataset.names], dtype=torch.float32) for k in LABEL_KEYS}

# ---- comparison graphics ---------------------------------------------------
# train AND val true-vs-predicted scatter per label (grids), per-label bar
# charts on the val split, and the shared embedding-norm histogram.
try:
    import matplotlib.pyplot as plt

    def preds_for(split_idx):
        out = {}
        for label_key in LABEL_KEYS:
            h = build_head()
            h.load_state_dict(torch.load(os.path.join(HEADS_DIR, f"{label_key}.pt"),
                                         map_location=device, weights_only=True)["state_dict"])
            h.eval()
            with torch.no_grad():
                out[label_key] = (ys_t_all[label_key][split_idx],
                                  h(Z[split_idx].to(device)).cpu().squeeze(-1))
        return out

    def plot_grid(preds, split_name):
        k = len(LABEL_KEYS)
        cols = min(3, k)
        rows = int(np.ceil(k / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
        axes = np.atleast_1d(axes).ravel()
        for ax, label_key in zip(axes, LABEL_KEYS):
            true, pred = preds[label_key]
            m = metrics[label_key]
            true = true * m["y_std"] + m["y_mean"]  # back to original units
            pred = pred * m["y_std"] + m["y_mean"]
            ax.scatter(true.numpy(), pred.numpy(), s=12)
            lo = float(min(true.min(), pred.min()))
            hi = float(max(true.max(), pred.max()))
            ax.plot([lo, hi], [lo, hi], "k--")
            r2_txt = f"{m['r2']:.3f}" if m["r2"] is not None else "n/a"
            ax.set_title(f"{label_key} (R2={r2_txt})", fontsize=9)
            ax.set_xlabel("true", fontsize=8)
            ax.set_ylabel("pred", fontsize=8)
        for ax in axes[k:]:
            ax.axis("off")
        fig.suptitle(f"per-label {split_name} predictions (true vs predicted)")
        fig.tight_layout()
        plt.show()

    plot_grid(preds_for(train_idx), "train")
    if n_val:
        plot_grid(preds_for(val_idx), "val")

    k = len(LABEL_KEYS)
    x = np.arange(k)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(x, [metrics[l]["rmse"] / max(metrics[l]["y_std"], 1e-12) for l in LABEL_KEYS])
    axes[0].set_xticks(x, LABEL_KEYS, rotation=45, ha="right", fontsize=8)
    axes[0].set_ylabel("val RMSE / train std")
    axes[0].set_title("normalized val RMSE per label")
    axes[1].bar(x, [metrics[l]["r2"] if metrics[l]["r2"] is not None else float("nan")
                    for l in LABEL_KEYS])  # nan = undefined (val split < 2 points)
    axes[1].set_xticks(x, LABEL_KEYS, rotation=45, ha="right", fontsize=8)
    axes[1].set_ylabel("R2")
    axes[1].set_title("val R2 per label")
    fig.tight_layout()
    plt.show()

    plt.figure(figsize=(5, 3.5))
    plt.hist(Z.norm(dim=1).numpy(), bins=30)
    plt.xlabel("embedding norm ||z||")
    plt.ylabel("count")
    plt.title("embedding norms (shared by all labels)")
    plt.show()
except ImportError:
    print("matplotlib not installed - skipping plots")
